# LNCLIP-DF — LayerNorm-tuned CLIP for Deepfake Detection

**Architecture:** CLIP ViT-L/14 + LN-tuning (0.03% params) + Angular Margin Loss + Compression Augmentation

**Based on:**
- Deepfake Detection that Generalizes Across Benchmarks (2025, arxiv 2508.06248)
- Unlocking Hidden Potential of CLIP (2025, arxiv 2503.19683)

**Dataset:** 1,056 videos (362 real + 694 fake) from 3 sources

**Runtime:** ~2h preprocessing + ~30min training on T4 GPU

---

**Before running:** Upload these folders to Google Drive at `MyDrive/VIPER/`:
- `dataset_production/` (580 videos)
- `dataset_week2/` (170 videos)
- `dfd_dataset/` (306 videos)

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 1 — Setup
# ═══════════════════════════════════════════════════════════════

import torch
assert torch.cuda.is_available(), 'GPU required → Runtime → Change runtime → T4'
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB)')

!pip install -q open_clip_torch insightface onnxruntime-gpu dlib
!pip install -q opencv-python-headless scipy scikit-learn tqdm matplotlib pandas
print('\n✓ Dependencies installed')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 2 — Mount Drive + Clone Repo
# ═══════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob
from pathlib import Path

if not os.path.exists('/content/VIPER'):
    !git clone https://github.com/rxbinsingh/VIPER /content/VIPER
else:
    !git -C /content/VIPER pull --quiet

os.chdir('/content/VIPER')
sys.path.insert(0, '/content/VIPER')

# Paths
BASE = '/content/drive/MyDrive/VIPER'
DATA_DIRS = [
    f'{BASE}/dataset_production',
    f'{BASE}/dataset_week2',
    f'{BASE}/dfd_dataset',
]
CACHE_DIR = f'{BASE}/cache_lnclip'
SAVE_DIR  = f'{BASE}/checkpoints_lnclip'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

# Verify datasets
total = 0
for d in DATA_DIRS:
    if os.path.exists(d):
        meta = f'{d}/metadata.csv'
        if os.path.exists(meta):
            import pandas as pd
            n = len(pd.read_csv(meta))
            total += n
            print(f'  ✓ {Path(d).name}: {n} videos')
        else:
            print(f'  ✗ {Path(d).name}: no metadata.csv')
    else:
        print(f'  ✗ {Path(d).name}: NOT FOUND')
print(f'\nTotal: {total} videos')
print(f'Cache: {CACHE_DIR}')
print(f'Saves: {SAVE_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 3 — Preprocess All Videos (cached to Drive)
# ═══════════════════════════════════════════════════════════════
# ~2 hours on CPU. Skips already-cached videos.
# Safe to restart — cache persists on Drive.

import cv2, numpy as np, pandas as pd
from tqdm import tqdm
from src.preprocessing import preprocess_video

# Patch BCR (mediapipe broken on 3.12)
import src.anchor_extractor as ae
ae.build_biomech_anchor = lambda frames: None

def get_all_videos():
    samples = []
    for d in DATA_DIRS:
        meta_path = f'{d}/metadata.csv'
        if not os.path.exists(meta_path): continue
        meta = pd.read_csv(meta_path)
        for _, row in meta.iterrows():
            label_str = row.get('label', 'unknown')
            label = 0 if label_str == 'real' else 1
            cat = row.get('category', label_str)
            fname = row['filename']
            vp = f'{d}/{cat}/{fname}'
            if not os.path.exists(vp):
                vp = f'{d}/{label_str}/{fname}'
            if os.path.exists(vp):
                samples.append((vp, label, cat))
    return samples

all_videos = get_all_videos()
already = len(glob.glob(f'{CACHE_DIR}/*.npz'))
print(f'Videos: {len(all_videos)} | Already cached: {already} | To process: {len(all_videos)-already}')

ok, fail = 0, 0
for vp, label, cat in tqdm(all_videos, desc='Preprocessing'):
    cp = f'{CACHE_DIR}/{Path(vp).stem}.npz'
    if os.path.exists(cp):
        ok += 1; continue
    try:
        p = preprocess_video(vp, num_frames=16, n_anchor=8)
        if not p['valid']:
            fail += 1; continue
        crops = [cv2.cvtColor(c, cv2.COLOR_BGR2RGB) for c in p['video_frames']]
        np.savez_compressed(cp, crops=np.stack(crops), label=np.array(label))
        ok += 1
    except:
        fail += 1

print(f'\nDone. OK: {ok} | Failed: {fail} | Success: {ok/(ok+fail)*100:.0f}%')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 4 — Build Model + Datasets
# ═══════════════════════════════════════════════════════════════

import torch, torch.nn as nn, torch.nn.functional as F
import random, numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms as T
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
from PIL import Image
import open_clip
from src.lnclip_model import build_lnclip_model

device = 'cuda'
random.seed(42); np.random.seed(42); torch.manual_seed(42)

# ── Compression Augmentation ──────────────────────────────────
class CompressAug:
    def __init__(self, p=0.5):
        self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        arr = np.array(img)
        if random.random() < 0.5:  # JPEG
            q = random.randint(30, 95)
            _, enc = cv2.imencode('.jpg', cv2.cvtColor(arr, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, q])
            arr = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        if random.random() < 0.3:  # Blur
            s = random.uniform(0.5, 2.0); k = int(s*4)|1
            arr = cv2.GaussianBlur(arr, (k,k), s)
        if random.random() < 0.3:  # Downscale
            sc = random.uniform(0.5, 0.9)
            h,w = arr.shape[:2]
            arr = cv2.resize(cv2.resize(arr, (int(w*sc),int(h*sc))), (w,h))
        return Image.fromarray(arr)

CLIP_TF = T.Compose([T.Resize(224), T.CenterCrop(224), T.ToTensor(),
    T.Normalize([0.48145466,0.4578275,0.40821073],[0.26862954,0.26130258,0.27577711])])

# ── Dataset ───────────────────────────────────────────────────
class LNCLIPDataset(Dataset):
    def __init__(self, samples, cache_dir, train=True):
        self.samples = [(p,l) for p,l,_ in samples if os.path.exists(f'{cache_dir}/{Path(p).stem}.npz')]
        self.cache_dir = cache_dir
        self.aug = CompressAug(0.5) if train else None
        self.flip = T.RandomHorizontalFlip(0.5) if train else T.RandomHorizontalFlip(0)
        print(f'  Dataset: {len(self.samples)} videos ({"train" if train else "eval"})')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        vp, label = self.samples[idx]
        data = np.load(f'{self.cache_dir}/{Path(vp).stem}.npz')
        crops = data['crops']
        T_actual = min(len(crops), 16)
        tensors = []
        for i in range(T_actual):
            img = Image.fromarray(crops[i])
            if self.aug: img = self.aug(img)
            img = self.flip(img)
            tensors.append(CLIP_TF(img))
        while len(tensors) < 16: tensors.append(tensors[-1])
        return {'crops': torch.stack(tensors[:16]), 'label': torch.tensor(label, dtype=torch.long)}

# ── Split ─────────────────────────────────────────────────────
all_videos_shuffled = all_videos.copy()
random.shuffle(all_videos_shuffled)
n = len(all_videos_shuffled)
train_s = all_videos_shuffled[:int(0.7*n)]
val_s   = all_videos_shuffled[int(0.7*n):int(0.8*n)]
test_s  = all_videos_shuffled[int(0.8*n):]
print(f'Split: Train={len(train_s)} Val={len(val_s)} Test={len(test_s)}')

train_ds = LNCLIPDataset(train_s, CACHE_DIR, train=True)
val_ds   = LNCLIPDataset(val_s, CACHE_DIR, train=False)
test_ds  = LNCLIPDataset(test_s, CACHE_DIR, train=False)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

# ── Model ─────────────────────────────────────────────────────
model, _ = build_lnclip_model(device=device, num_trainable_layers=6,
                               arc_scale=30.0, arc_margin=0.3, sphere_noise=0.05)
print(f'\nModel ready. Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} params')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 5 — Training (~30 min)
# ═══════════════════════════════════════════════════════════════

EPOCHS = 20
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

def train_epoch(model, loader):
    model.train()
    all_l, all_p, total_loss = [], [], 0.0
    for batch in tqdm(loader, leave=False, desc='Train'):
        crops, labels = batch['crops'].to(device), batch['label'].to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits, _ = model(crops, labels)
            loss = F.cross_entropy(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
        all_l.extend(labels.cpu().numpy())
        all_p.extend(F.softmax(logits.detach(), dim=-1)[:,1].cpu().numpy())
    return total_loss/len(loader), roc_auc_score(all_l, all_p) if len(set(all_l))>1 else 0

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    all_l, all_p = [], []
    for batch in tqdm(loader, leave=False, desc='Eval'):
        crops = batch['crops'].to(device)
        with torch.amp.autocast('cuda'):
            logits, _ = model(crops, labels=None)
        all_l.extend(batch['label'].numpy())
        all_p.extend(F.softmax(logits, dim=-1)[:,1].cpu().numpy())
    auc = roc_auc_score(all_l, all_p) if len(set(all_l))>1 else 0
    preds = [1 if p>0.5 else 0 for p in all_p]
    return auc, accuracy_score(all_l, preds), all_l, all_p

best_auc, patience_count = 0.0, 0
PATIENCE = 7
print(f'Training LNCLIP-DF for up to {EPOCHS} epochs (patience={PATIENCE})...\n')

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_auc = train_epoch(model, train_loader)
    vl_auc, vl_acc, _, _ = eval_epoch(model, val_loader)
    scheduler.step()
    flag = ''
    if vl_auc > best_auc:
        best_auc = vl_auc; patience_count = 0
        torch.save(model.state_dict(), f'{SAVE_DIR}/lnclip_best.pt')
        flag = '  ← best'
    else:
        patience_count += 1
    print(f'Epoch {epoch:02d}/{EPOCHS}  Loss={tr_loss:.4f}  Train={tr_auc:.4f}  Val={vl_auc:.4f}  Acc={vl_acc:.4f}{flag}')
    if patience_count >= PATIENCE:
        print(f'Early stopping at epoch {epoch}'); break

print(f'\nBest Val AUC: {best_auc:.4f}')
torch.save(model.state_dict(), f'{SAVE_DIR}/lnclip_final.pt')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 6 — Test Evaluation + TTA
# ═══════════════════════════════════════════════════════════════

model.load_state_dict(torch.load(f'{SAVE_DIR}/lnclip_best.pt', map_location=device))
model.eval()

# TTA: original + horizontal flip
all_labels, all_probs = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test + TTA'):
        crops = batch['crops'].to(device)
        with torch.amp.autocast('cuda'):
            logits1, _ = model(crops, labels=None)
            logits2, _ = model(torch.flip(crops, [-1]), labels=None)
        probs = (F.softmax(logits1, dim=-1)[:,1] + F.softmax(logits2, dim=-1)[:,1]) / 2
        all_labels.extend(batch['label'].numpy())
        all_probs.extend(probs.cpu().numpy())

auc = roc_auc_score(all_labels, all_probs)
preds = [1 if p>0.5 else 0 for p in all_probs]
acc = accuracy_score(all_labels, preds)
cm = confusion_matrix(all_labels, preds)

print(f'\n{"═"*55}')
print(f'  LNCLIP-DF — FINAL TEST RESULTS')
print(f'{"═"*55}')
print(f'  AUC-ROC:  {auc:.4f}')
print(f'  Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print(f'\n  Confusion Matrix:')
print(f'    TN={cm[0,0]}  FP={cm[0,1]}')
print(f'    FN={cm[1,0]}  TP={cm[1,1]}')
print(f'\n{classification_report(all_labels, preds, target_names=["Real","Fake"])}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 7 — Per-Category Breakdown
# ═══════════════════════════════════════════════════════════════

cat_labels = {}
cat_probs = {}

for (vp, label, cat), prob in zip(test_s, all_probs[:len(test_s)]):
    cp = f'{CACHE_DIR}/{Path(vp).stem}.npz'
    if not os.path.exists(cp): continue
    if cat not in cat_labels:
        cat_labels[cat] = []
        cat_probs[cat] = []
    cat_labels[cat].append(label)
    cat_probs[cat].append(prob)

print(f'{"Category":<25} {"AUC":>8} {"N":>5}')
print('─'*40)
real_l = cat_labels.get('real', [])
real_p = cat_probs.get('real', [])
for cat in sorted(cat_labels.keys()):
    if cat == 'real': continue
    cl = real_l + cat_labels[cat]
    cp_ = real_p + cat_probs[cat]
    if len(set(cl)) < 2: continue
    a = roc_auc_score(cl, cp_)
    print(f'{cat:<25} {a:>8.4f} {len(cat_labels[cat]):>5}')
print('─'*40)
print(f'{"ALL":<25} {auc:>8.4f} {len(all_labels):>5}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CELL 8 — Save Results
# ═══════════════════════════════════════════════════════════════

import json
results = {
    'model': 'LNCLIP-DF (LN-tuned CLIP ViT-L/14 + Angular Margin)',
    'test_auc': round(auc, 4),
    'test_accuracy': round(acc, 4),
    'best_val_auc': round(best_auc, 4),
    'total_videos': len(all_videos),
    'trainable_params_pct': '0.03%',
    'confusion_matrix': cm.tolist(),
    'architecture': {
        'backbone': 'CLIP ViT-L/14 (OpenAI)',
        'tuning': 'LayerNorm only (last 6 transformer layers)',
        'loss': 'Angular Margin (ArcFace-style, scale=30, margin=0.3)',
        'augmentation': 'Compression (JPEG/blur/resize) + horizontal flip',
        'temporal': 'Mean pool 16 frames',
        'normalization': 'L2 (unit hypersphere)',
    },
}
with open(f'{SAVE_DIR}/results_lnclip.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Results saved.')
print(json.dumps(results, indent=2))